## Virtual Staining with ImmuVis

The code below shows and example of how to perform virtual staining with ImmuVis models.

## 0. Installation

To install the ImmuVis dependencies, we recommend creating a virtual environment using [uv](https://docs.astral.sh/uv/getting-started/installation/):

```bash
cd ImmuVis
uv sync --python 3.12 --extra notebooks
```
Register the notebook kernel:

```bash
uv run python -m ipykernel install --user --name immuvis --display-name "Python (ImmuVis)"
```

Select the **Python (ImmuVis)** kernel. The model-loading cell below downloads the model and its matching tokenizer from Hugging Face; no local checkpoint or tokenizer file is needed.

## 1. Data preparation

For Virtual Staining, you will need your IMC images arranged in the following directory structure:

```text
.
└── data
    ├── test
    │   ├── dataset1
    │   │   └── imgs
    │   ├── dataset2
    │   │   └── imgs
    │   └── dataset3
    │       └── imgs
    └── train
        ├── dataset1
        │   └── imgs
        ├── dataset2
        │   └── imgs
        └── dataset3
            └── imgs
```

where `test` and `train` are the names of a split - if your data does not need any such splitting, you can consolidate it under one directory, say `all`:

```text
.
└── data
    └── all
        ├── dataset1
        │   └── imgs
        ├── dataset2
        │   └── imgs
        └── dataset3
            └── imgs
```
`dataset1` etc are the custom names of your datasets/panels - the assumption is that all the IMC images (either as .tiff or .npy) under `imgs` for a given dataset have exactly the same panel composition (equal number and order of channels). If your dataset consists of separate panels (with different marker subsets measured) split your images into separate dataset directories, like `dataset1-panelA`, `dataset1-panelB` etc. Note that `imgs` must be a subdirectory in all these dataset folders.

For ImmuVis native dataset, you need to provide a configuration file specifying the following fields:
1. `paths` with a mapping from split to absolute filesystem path
2. `datasets` listing all the datasets within a split you want to use
3. `clip_limits` - for each dataset, provide a dataset-level 99th percentile value of all its corresponding IMC images pixels
4. `markers` with a mapping from a dataset to a list of marker names, corresponding the image channel at each position, i.e. marker name at index 0 corresponds to marker measured at the channel 0 of an image.
   
**IMPORTANT** The provided marker names must EXACTLY match the supported marker names in `TOKENIZER_CONFIG`, downloaded with the Hugging Face model below. If your dataset has some unsupported markers, you still need to provide their names in proper order, but their corresponding image channels will be skipped during loading. If you would like to be able to process these markers, you can run lightweight finetuning script (See the last cell in this notebook for details).

Below is the dummy example of such config - it can be either typed directly in the code or loaded from e.g. YAML file:

In [1]:
DATASET_CONFIG = {
    # Paths to the data splits
    "paths": {
        "all": "/path/to/IMC/all",
    },

    # Dataset names inside the splits paths to use
    "datasets": [
        "nsclc2-panel1",
        "nsclc2-panel2",
    ],

    # 99th percentile pixel values for each dataset
    "clip_limits": {
        "nsclc2-panel1": 3.1,
        "nsclc2-panel2": 2.7,
    },

    # Ordered list of markers measured in each dataset
    "markers": {
        "nsclc2-panel1": [
            "CD45",
            "CD3",
            "CD4",
            "CD8a",
            "CD20",
            "CD68",
            "PD1",
            "PDL1",
            "FOXP3",
            "panCK",
        ],
        "nsclc2-panel2": [
            "DNA1",
            "DNA2",
            "CD4",
            "CD8a",
            "CD137",
            "Ecad",
            "PDL1",
            "FOXP3",
            "Sox10",
        ],
    }
}

Load the ImmuVis model and its matching marker tokenizer together. Set `MODEL_REPO_ID` to `SzczurekLab/ImmuVis_Conv`, `SzczurekLab/ImmuVis_Swin`, or `SzczurekLab/ImmuVis_ViT`. All three original variants share the same 265-marker mapping. Set `MODEL_REVISION` to a commit hash to pin a release, or leave it as `None` for the default branch. Rerun this cell and all following cells after switching models.

In [ ]:
import torch
from multiplex_model.modules import MultiplexAutoencoder

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

MODEL_REPO_ID = "SzczurekLab/ImmuVis_Conv"
MODEL_REVISION = None
model = MultiplexAutoencoder.from_pretrained(
    MODEL_REPO_ID,
    revision=MODEL_REVISION,
    map_location="cpu",
).to(DEVICE)

TOKENIZER_CONFIG = model.marker_tokenizer

print(f"Loaded {MODEL_REPO_ID} on {DEVICE}; tokenizer vocabulary: {len(TOKENIZER_CONFIG)} markers")

Ininialize the `MultiplexDataset` (natively handles all expected ImmuVis preprocessing and tokenization by default):

In [ ]:
from multiplex_model.data import MultiplexDataset, PanelBatchSampler, TestCrop
from torch.utils.data import DataLoader

BATCH_SIZE = 8
transform = TestCrop(size=128) # central crop of 128x128 pixels

# Create the dataset
dataset = MultiplexDataset(
    panels_config=DATASET_CONFIG,
    split="all", # or "train", "test" etc.
    marker_tokenizer=TOKENIZER_CONFIG,
    unsupported_marker_behavior="drop", # omit unsupported markers
    transform=transform,
    file_extension="tiff" # or "npy" if your data is in numpy array format
)

# Sampler for batches - one batch is sampled from a single dataset, 
# so the channel dimensions are consistent within a batch
batch_sampler = PanelBatchSampler(
    dataset=dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# Create the dataloader
dataloader = DataLoader(
    dataset,
    batch_sampler=batch_sampler,
    shuffle=False,
)

## 2. Inspect the loaded model

The Hugging Face model has already been loaded alongside the tokenizer before dataset construction. Its saved architecture is reconstructed automatically, and it is in evaluation mode on `DEVICE`. Change `MODEL_REPO_ID` in the loading cell above and rerun the following cells to select Conv, Swin, or ViT.

In [ ]:
print(f"Loaded {MODEL_REPO_ID}; latent dimension: {model.latent_dim}")

## 3. Perform virtual staining

Determine what markers you want to decode (predict from the existing ones) and tokenize them:

In [ ]:
decoded_markers = [
    "CXCL4",
    "CD34",
    "GHRL"
]

def tokenize_markers(markers, tokenizer):
    """
    Convert a list of marker names to their corresponding token IDs using the provided tokenizer.
    """
    token_ids = []
    for marker in markers:
        if marker in tokenizer:
            token_ids.append(tokenizer[marker])
        else:
            print(f"Warning: Marker '{marker}' not found in tokenizer. It will be skipped.")
    return token_ids

decoded_marker_tokens = tokenize_markers(decoded_markers, TOKENIZER_CONFIG)
decoded_marker_tokens

In [ ]:
# Add batch dimension
decoded_ids_batch = (
    torch.tensor(decoded_marker_tokens, dtype=torch.long)
    .unsqueeze(0)
    .repeat_interleave(BATCH_SIZE, dim=0)
    .to(DEVICE)
) 

decoded_ids_batch

Now you can iterate over the datasets with dataloader and perform virtual staining of selected markers:

In [ ]:
staining_predictions = []
staining_uncertainties = []

for img, channel_ids, dataset_name, img_path in dataloader:
    img = img.to(DEVICE)
    channel_ids = channel_ids.to(DEVICE)

    with torch.no_grad():
        # Forward pass through the model
        outputs = model(img, channel_ids, decoded_ids_batch)["output"]

        # Model returns mean and log variance per pixel
        means, log_vars = outputs.unbind(-1)

        predictions = torch.sigmoid(means)
        uncertainties = torch.exp(log_vars)

        staining_predictions.append(predictions.cpu())
        staining_uncertainties.append(uncertainties.cpu())

## \* Finetuning for unseen markers support

One way of dealing with unseen markers with ImmuVis is to add their corresponding embeddings to ImmuVis hyperkernels and finetune them, with the rest of the backbone frozen. To do so, create a matching finetuning config (see `finetuning_configs/finetune_masked_config.yaml`) and simply run the finetuning script:

```bash
python3 finetune_masked_model.py finetuning_configs/finetune_masked_config.yaml
```

See README for setup details (the same as for training).